# Generative AI 007 — Structured Output

Defining a schema is pure Python, so **all of it can be checked without an API
key**. The `with_structured_output` call itself cannot — see Part F.

| Part | What we check |
|---|---|
| A | a TypedDict accepts **all four** deliberately broken records |
| B | Pydantic rejects them, and coerces `"32"` to `32` |
| C | the `Annotated` trap — your description lands under **`default`** |
| D | no docstring ships **370 chars** of `dict`'s own documentation |
| E | the two styles send **byte-identical** schemas for required fields |
| F | why the call itself is not measured here |

Needs `langchain-core`, `pydantic`, `typing_extensions`.

In [ ]:
import warnings, json
warnings.filterwarnings("ignore")

from typing import Annotated, Literal, Optional
from typing_extensions import TypedDict
from pydantic import BaseModel, Field, ValidationError
from langchain_core.utils.function_calling import convert_to_openai_tool

import langchain_core
print("langchain_core", langchain_core.__version__)

## Part A — A TypedDict checks nothing

A `TypedDict` declares which keys a dictionary should have and what type each
value should be. At run time it is a **plain `dict`** — the annotations were
erased, and nothing looks at them.

In [ ]:
class PersonTD(TypedDict):
    name: str
    age: int

broken = [
    {"name": "Ayesha", "age": "not a number"},   # str where int declared
    {"name": 123, "age": 30},                    # int where str declared
    {"name": "Ayesha", "age": "32"},             # numeric string
    {"name": "Ayesha"},                          # missing field
]

for rec in broken:
    p: PersonTD = rec        # type-annotated assignment; Python does not check it
    print("accepted:", p)

print("\ntype of a TypedDict instance:", type(broken[0]).__name__)
assert type(broken[0]) is dict

Four for four. `{"name": 123, "age": "not a number"}` is a perfectly valid
dictionary as far as Python is concerned.

That is not a bug — it is what a TypedDict *is*. The annotations are for your
editor and your type checker, and they vanish at run time.

## Part B — Pydantic checks everything

In [ ]:
class PersonPD(BaseModel):
    name: str
    age: int

for rec in broken:
    try:
        obj = PersonPD(**rec)
        print(f"accepted -> age={obj.age!r} ({type(obj.age).__name__})")
    except ValidationError as e:
        print(f"rejected -> {e.errors()[0]['type']}")

In [ ]:
# The third record is the interesting one: "32" was COERCED to the integer 32.
print(repr(PersonPD(name="A", age="32").age))
assert PersonPD(name="A", age="32").age == 32
assert isinstance(PersonPD(name="A", age="32").age, int)

In [ ]:
# And the part a TypedDict cannot express at all: a RULE about the value.
class Student(BaseModel):
    """A student record."""
    name: str = "Anonymous"
    age: Optional[int] = None
    cgpa: float = Field(gt=0, lt=10, default=5.0, description="The CGPA")

print("defaults :", Student().model_dump())
print("coercion :", repr(Student(cgpa="7.5").cgpa))
for bad in (11, -1):
    try:
        Student(cgpa=bad)
    except ValidationError as e:
        print(f"cgpa={bad:<4} -> {e.errors()[0]['type']}: {e.errors()[0]['msg']}")

print("as JSON  :", Student(name="Ayesha", age=21).model_dump_json())

"The CGPA must be between 0 and 10" is a **rule**, not a type. There is no
annotation that says it. If your schema has any constraint of that kind,
Pydantic is the only one of the three styles that can state it.

## Part C — The `Annotated` trap

Descriptions are how you steer the model. In a TypedDict you attach one with
`Annotated`. Almost every tutorial writes it with two arguments.

In [ ]:
def summary_schema(cls):
    return convert_to_openai_tool(cls)["function"]["parameters"]["properties"]["summary"]

class TwoArg(TypedDict):
    summary: Annotated[str, "A brief summary of the review"]

class ThreeArg(TypedDict):
    summary: Annotated[str, ..., "A brief summary of the review"]

print("two args  ->", json.dumps(summary_schema(TwoArg)))
print("three args->", json.dumps(summary_schema(ThreeArg)))

assert "default" in summary_schema(TwoArg)
assert "description" not in summary_schema(TwoArg)
assert "description" in summary_schema(ThreeArg)

**Read the key, not the text.**

The two-argument form filed your sentence under **`default`** — a value to use
when the field is missing. The model receives **no description at all**, and is
told the default value of `summary` is the English sentence *"A brief summary of
the review"*.

Nothing raises. You get worse extractions and no clue why.

The middle slot is the default, so write `...` there to say there is not one.

## Part D — Give the schema a docstring

In [ ]:
class NoDocstring(TypedDict):
    summary: Annotated[str, ..., "A brief summary"]

class WithDocstring(TypedDict):
    """Structured insights extracted from a product review."""
    summary: Annotated[str, ..., "A brief summary"]

for cls in (NoDocstring, WithDocstring):
    desc = convert_to_openai_tool(cls)["function"]["description"]
    print(f"{cls.__name__:<14} {len(desc):>4} chars | {desc[:58]!r}")

nodoc = convert_to_openai_tool(NoDocstring)["function"]["description"]
assert len(nodoc) == 370 and nodoc.startswith("dict()")

With no docstring the schema inherits **`dict`'s own documentation** and ships
370 characters of it to the model as the description of your extraction task.

Harmless, useless, and billed on every call.

## Part E — Do the styles actually send different schemas?

This is the question the usual feature table does not answer.

In [ ]:
class ReviewTD(TypedDict):
    """Structured insights extracted from a product review."""
    key_themes: Annotated[list[str], ..., "All key themes discussed, as a list"]
    summary: Annotated[str, ..., "A brief summary of the review"]
    sentiment: Annotated[Literal["pos", "neg"], ..., "Overall sentiment"]
    pros: Annotated[Optional[list[str]], None, "All pros, as a list"]

class ReviewPD(BaseModel):
    """Structured insights extracted from a product review."""
    key_themes: list[str] = Field(description="All key themes discussed, as a list")
    summary: str = Field(description="A brief summary of the review")
    sentiment: Literal["pos", "neg"] = Field(description="Overall sentiment")
    pros: Optional[list[str]] = Field(default=None, description="All pros, as a list")

td = convert_to_openai_tool(ReviewTD)["function"]
pd = convert_to_openai_tool(ReviewPD)["function"]

for name in td["parameters"]["properties"]:
    same = td["parameters"]["properties"][name] == pd["parameters"]["properties"][name]
    print(f"{name:<12} agree: {same}")

print("\nrequired (both):", td["parameters"]["required"])
print("description same:", td["description"] == pd["description"])

In [ ]:
# The one disagreement, in full:
print("TypedDict:", json.dumps(td["parameters"]["properties"]["pros"]))
print("Pydantic :", json.dumps(pd["parameters"]["properties"]["pros"]))

# Pydantic says "array OR null, defaulting to null".
# The TypedDict route says "array" - the nullability was DROPPED.
assert "anyOf" in pd["parameters"]["properties"]["pros"]
assert "anyOf" not in td["parameters"]["properties"]["pros"]

# Every REQUIRED field is byte-identical:
for name in ("key_themes", "summary", "sentiment"):
    assert td["parameters"]["properties"][name] == pd["parameters"]["properties"][name]
print("\nall three required fields byte-identical: True")

So the choice between TypedDict and Pydantic barely changes **what the model is
asked to do**. It entirely changes **what happens when the answer comes back**.

Choose on that — which means Pydantic, almost always.

## Part F — What cannot be measured here

In [ ]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel

model = FakeListChatModel(responses=['{"summary": "good", "sentiment": "pos"}'])
try:
    model.with_structured_output(ReviewPD)
    print("accepted")
except NotImplementedError as e:
    print("NotImplementedError:", e)

print()
print("So this notebook cannot measure:")
print("  - whether a given model actually obeys the schema")
print("  - how often it fails")
print("  - whether json_mode or function_calling does better")
print()
print("Those need a real provider and a paid key. Everything above is about")
print("the schema you SEND, which is settled before any model is involved.")

## What to take away

- **A TypedDict validates nothing** — all four broken records were accepted.
- **Pydantic validates, coerces `"32"` to `32`, and enforces `gt`/`lt`** rules
  that no annotation can express.
- **Write `Annotated[str, ..., "desc"]`** — the two-argument form files your
  description under `default` and the model never sees it.
- **Give every schema a docstring**, or it sends 370 characters of `dict`'s
  documentation instead.
- **Required fields produce byte-identical schemas** either way; only optional
  fields differ, and there TypedDict drops the nullability.

## Exercises

1. Add a fifth broken record — a list where a string was declared. Does
   Pydantic's error type differ from the ones you saw?
2. Pydantic coerced `"32"` to `32`. Find a value where coercion is *not* what
   you want, and look up `model_config = ConfigDict(strict=True)`.
3. Part E compared two styles. Write the same schema as raw JSON Schema and
   compare all three. Which fields does the hand-written version get wrong first?
4. The `Annotated` trap is silent. Write a check you could run in a test suite
   that fails when any field in a schema has a `default` that looks like prose.
5. If you have an API key: run the same schema through `json_mode` and
   `function_calling` twenty times each and count the failures. That is the
   measurement this notebook could not make.